# Basic cleaning


This NB reads the raw corpus (6 manifestos, in Spanish), repairs PDF-extraction artifacts, and saves the result to a new CSV (`text_clean`).
It does **not** lemmatize, lowercase, or remove stopwords -- those are linguistic decisions that belong to whichever notebook comes next. The point is that all downstream pipelines start from the same repaired text instead of each one reinventing its own cleanup.

Every cleaning step below is tagged:
- **[GENERAL]**: applies to any PDF-extracted corpus, in any language.
- **[THIS CORPUS]**: specific to these 6 Peruvian "planes de gobierno"   (digital-signature blocks, repeated headers). If you reuse this notebook  on a different corpus, inspect your own raw documents before assuming   these patterns transfer.


In [1]:
import re
import unicodedata
import pandas as pd

CORPUS_URL = "https://github.com/eScience-SummerSchool/manifestos/raw/main/PROCESSED/corpus_raw.csv"
corpus = pd.read_csv(CORPUS_URL)
corpus.info()
corpus[["party", "n_characters", "n_words"]]


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6 entries, 0 to 5
Data columns (total 4 columns):
 #   Column        Non-Null Count  Dtype 
---  ------        --------------  ----- 
 0   party         6 non-null      object
 1   text_raw      6 non-null      object
 2   n_characters  6 non-null      int64 
 3   n_words       6 non-null      int64 
dtypes: int64(2), object(2)
memory usage: 324.0+ bytes


,party,n_characters,n_words
0,AhoraNacion,373258,53720
1,BuenGobierno,221197,28149
2,FuerzaPopular,327610,44266
3,JuntosPorElPeru,215489,30513
4,Obras,40765,5583
5,RenovacionPopular,51803,6868


## 1. Particularities

Confirming what we already know is there


In [ ]:
print(corpus.loc[0, "party"])
print("-" * 60)
print(corpus.loc[0, "text_raw"][:600])


## 2. Patterns specific to this corpus

The digital-signature footer is one contiguous block in the source text:
`"Firmado Digitalmente por:"` -> a line with the signer's name -> a
certificate/hash line (e.g. `"GIDO FIR 70780350 hard"`) -> `"Fecha: ..."`.
This was found by manually inspecting `corpus.loc[0, "text_raw"]`, not
assumed generically -- if you point this notebook at a different corpus,
re-derive this list from your own documents.


In [ ]:
## claude recommendation

BOILERPLATE_PATTERNS = [
    # [THIS CORPUS] The whole signature footer as one unit (non-greedy,
    # spanning newlines). This is the source of most of the "junk tokens"
    # (DNI-like numbers, "firmado", "digitalmente") seen downstream if this
    # step is skipped.
    r"Firmado\s+Digitalmente\s+por:?.*?Fecha:\s*\d{1,2}/\d{1,2}/\d{4}\s+\d{1,2}:\d{2}:\d{2}",
    # [THIS CORPUS] Defensive fallback: catches an 8-digit Peruvian DNI
    # number if one ever appears OUTSIDE the signature block above (e.g.
    # cited in running text rather than only in the footer).
    r"\bDNI\s*[:\-]?\s*\d{8}\b",
    # [THIS CORPUS] The cover-page title "PLAN DE GOBIERNO (2026-2031)" --
    # requires a year range to be present, so it does NOT match the phrase
    # "plan de gobierno" when it shows up in normal prose without a year
    # attached (e.g. "nuestro plan de gobierno propone..."). Handles both
    # variants seen in this corpus (with/without parentheses) and the case
    # where the title is split across two lines in the raw PDF extraction
    # ("PLAN DE GOBIERNO" on one line, "(2026-2031)" on the next) -- \s*
    # matches the newline between them just like a space. Removes the
    # title outright rather than relying on the header_min_repeats dedup,
    # which only collapses duplicates to one surviving copy, not zero.
    r"PLAN\s+DE\s+GOBIERNO\s*\(?\s*\d{4}\s*-\s*\d{4}\s*\)?",
]


## 3. The cleaning function

Only repairs extraction damage. Deliberately does NOT lowercase, strip
punctuation, or remove stopwords -- that's left for later steps.

In [ ]:

def clean_pdf_text(text, boilerplate_patterns=BOILERPLATE_PATTERNS, header_min_repeats=3):
    """
    Repair PDF-extraction artifacts in a single document's raw text.

    Parameters
    ----------
    text : str
        One row of `text_raw` -- a full document as extracted from PDF.
    boilerplate_patterns : list[str]
        Regexes for corpus-specific junk blocks (signature footers, IDs,
        etc). Swap this list out entirely when you point this notebook at a
        different corpus.
    header_min_repeats : int
        A line that repeats at least this many times inside the SAME
        document is treated as a running header/footer and collapsed to a
        single occurrence. [GENERAL] mechanism; the actual repeated text
        (e.g. "PLAN DE GOBIERNO (2026-2031)") is [THIS CORPUS].

    Returns
    -------
    str : cleaned text, still cased and punctuated -- ready to hand to an
    LLM prompt, a spaCy pipeline, or an embeddings model. Not ready for a
    DTM by itself.
    """

    # [GENERAL] Unicode-normalize first, before any regex runs on the
    # string. PDF extraction can leave decomposed accents or compatibility
    # variants of the same visual character; NFKC folds them to one
    # canonical form so later regexes (and whatever tokenizer you use next)
    # don't silently miss matches because "e" + combining accent != "é" as
    # a single codepoint. This matters a lot for Spanish text specifically,
    # given how many accented characters it has.
    text = unicodedata.normalize("NFKC", text)

    # [GENERAL] Replace non-breaking spaces (U+00A0) and other invisible
    # unicode whitespace with a normal space. These are common in PDF-to-
    # text output and aren't always matched by a plain "\s" in every regex
    # engine configuration, so leaving them in causes silent tokenization
    # bugs later.
    text = re.sub(r"[  -​]", " ", text)

    # [GENERAL] Normalize curly/smart quotes and dash variants to their
    # plain-ASCII equivalents. Doesn't change meaning, just avoids the LLM,
    # spaCy, or an embedding model's tokenizer treating a curly quote and a
    # straight quote as different punctuation contexts.
    text = text.replace("“", '"').replace("”", '"')  # [GENERAL] left/right curly double quotes -> straight
    text = text.replace("‘", "'").replace("’", "'")  # [GENERAL] left/right curly single quotes -> straight
    text = text.replace("–", "-").replace("—", "-")  # [GENERAL] en-dash/em-dash -> hyphen

    # [GENERAL] Rejoin words that were split by a line-break hyphen during
    # PDF extraction, e.g. "gobier-\nno" -> "gobierno". Must run BEFORE the
    # whitespace-collapsing step below, since it depends on the literal
    # newline still being present right after the hyphen. This is a
    # heuristic: it will occasionally misjoin a legitimately hyphenated
    # compound that happened to fall at a line break -- spot-check a sample
    # after running this.
    text = re.sub(r"(\w)-\n(\w)", r"\1\2", text)

    # [THIS CORPUS] Strip the digital-signature / certificate boilerplate
    # found in these documents.
    for pattern in boilerplate_patterns:
        # re.DOTALL: the signature-block pattern spans multiple lines, so
        # "." must match newlines too, or the non-greedy .*? would stop at
        # the first line break instead of reaching the "Fecha:" line.
        text = re.sub(pattern, " ", text, flags=re.IGNORECASE | re.DOTALL)

    # [GENERAL, heuristic] Drop lines that consist of nothing but a short
    # digit run -- almost always a page number in PDF-extracted text. The
    # 1-3 digit cutoff is a reasonable default for a document under ~999
    # pages; verify against your own corpus if documents are longer or
    # number pages differently.
    text = re.sub(r"(?m)^\s*\d{1,3}\s*$", "", text)

    # [GENERAL] Strip table-of-contents "dot leaders" -- the run of periods
    # PDF extraction produces to visually connect a section title to its
    # page number (e.g. "PROPUESTAS PARA LA SALUD .......... 15") -- along
    # with the trailing page number, while KEEPING the section title text.
    # The title words are often genuinely useful vocabulary for a DTM (they
    # tell you the document has a health-policy section, an agriculture
    # section, etc), so this only removes the dots-plus-number tail, not
    # the whole line. The 4-dot minimum is comfortably above a real
    # ellipsis ("..." is 3 dots) and comfortably below the dozens of dots a
    # real dot leader produces, so normal prose with an ellipsis survives.
    # Known limitation: if this exact title also appears later as the real
    # section header when that section actually starts, the title words
    # now appear twice in the document (once from the TOC, once from the
    # real header) -- a small, bounded duplication, not the page-by-page
    # inflation a repeated running header would cause. If you'd rather
    # drop the TOC entry entirely instead of keeping the title, change the
    # replacement pattern below to strip the whole line.
    text = re.sub(r"(?m)\.{4,}\s*\d{1,4}\s*$", "", text)

    # [GENERAL] Collapse any line that repeats >= header_min_repeats times
    # within this SAME document -- typically a running header/footer that
    # PDF extraction duplicates once per page. Left unhandled, these inflate
    # raw term frequency for a DTM (or bias a pooled embedding) in a way
    # that has nothing to do with actual content.
    lines = text.split("\n")
    line_counts = {}
    for ln in lines:
        key = ln.strip()
        if key:  # [GENERAL] ignore blank lines for the repeat count
            line_counts[key] = line_counts.get(key, 0) + 1
    seen_repeated = set()
    deduped_lines = []
    for ln in lines:
        key = ln.strip()
        if key and line_counts.get(key, 0) >= header_min_repeats:
            if key in seen_repeated:
                continue  # [GENERAL] drop the 2nd, 3rd, ... occurrence
            seen_repeated.add(key)  # [GENERAL] keep the first occurrence
        deduped_lines.append(ln)
    text = "\n".join(deduped_lines)

    # [GENERAL] Collapse runs of blank lines and repeated whitespace (the
    # visual-spacing padding common in PDF extraction) down to a single
    # space, then trim the ends. Done LAST so it doesn't interfere with the
    # newline-dependent steps above (dehyphenation, header dedup).
    text = re.sub(r"\s+", " ", text).strip()

    return text


## 4. Quick test

Just for `Ahora Nacion`

In [ ]:
clean_pdf_text(corpus.loc[0, "text_raw"])

## 5. Apply to all 6 documents and save a sanity check

`pct_removed` per party is itself a diagnostic: it tells you how much of
the "raw" text was actually extraction noise, not manifesto content.


In [ ]:
def clean_corpus(corpus_df, text_col="text_raw"):
    cleaned = corpus_df.copy()
    cleaned["text_clean"] = cleaned[text_col].apply(clean_pdf_text)
    cleaned["n_characters_clean"] = cleaned["text_clean"].str.len()
    cleaned["n_words_clean"] = cleaned["text_clean"].str.split().apply(len)
    cleaned["n_characters_removed"] = cleaned[text_col].str.len() - cleaned["n_characters_clean"]
    cleaned["pct_removed"] = (100 * cleaned["n_characters_removed"] / cleaned[text_col].str.len()).round(1)
    return cleaned

clean_df = clean_corpus(corpus)
clean_df[["party", "n_characters", "n_characters_clean", "n_characters_removed", "pct_removed", "n_words", "n_words_clean"]]


Check this table before saving: if `pct_removed` is suspiciously high for
some party (document almost emptied out) or suspiciously low for a
document you know has a signature block, that's a signal the
`[THIS CORPUS]` patterns need adjusting for that particular party.


## 6. Save the output

This CSV is the shared starting point for whichever comes next:.
